# Pooling dose-response: does unique-to-structure decay as sites are added?

**Reconstruction.** The published `results/pooling_dose_response.json` — the
source of the write-up's §05 chart (race 11.5% → 2.7%, 77% removed, p=2e-8) —
was generated ad hoc on 2026-08-22 and the script was never saved. The raw
per-draw records survived in `out/_pooling_dose_perdraw.json` (720 rows), and
every published statistic recomputes from them exactly, so the numbers are
auditable. This notebook rebuilds the *generator*.

**Design (as published).** eICU-CRD, matched 12-analyte panel, k=5, one shared
codebook fit on the full pooled cohort. Total n held fixed at 3,000 while the
number of contributing hospitals `m` varies over 1, 2, 4, 8, 16, 32, drawing
`3000/m` patients from each. Holding n fixed is what separates "pooling across
sites" from "more data". 20 hospital draws x 3 codebook seeds per m.

**What the reconstruction cannot recover.** The per-draw file stores only
`(target, m, u_str, syn, u_val)` — no hospital ids, no RNG seed, no record of
the selection rule. So this reproduces the design, not the draws: expect
statistically equivalent, not identical, numbers.

**Two things this notebook adds, because the rebuild surfaced them.**

1. **The eligible hospital pool changes with m, and at m=1 it is tiny.** A draw
   at `m` needs hospitals holding at least `3000/m` patients. Only **3**
   hospitals clear 3,000 for mortality — and only **2** for race, whose contrast
   drops the Other/Unknown ethnicity buckets. So the m=1 anchor of the headline
   "11.5% -> 2.7%" is a two-hospital average, while m=32 draws from 155. The
   site *population* varies along the x-axis, not only the site *count*, and the
   published design note ("only the number of sites changes") is not strictly
   true. This also explains the bimodality in the surviving draws: race at m=1
   has 25/60 records at ~0 and 34/60 above 10, which is what two hospitals look
   like when one of them carries the signal and the other does not.

2. **A truncation sensitivity.** Because of (1), the slope is refit dropping the
   low-m points. Race survives this; mortality does not — see the last section.

In [1]:
import sys
sys.path.insert(0, "../src")

import json
import time
import zlib

import numpy as np
import pandas as pd
from scipy import stats

import eicu_channels as eicu
import pid_broja as pb
import quantize as qz

pd.set_option("display.width", 200)

K = 5                                  # forced by n=3000 per draw, NOT the headline k=13
SEEDS = [0, 1, 2]                      # codebook seeds
N_TOTAL = 3000                         # held FIXED across m -- the whole point
M_GRID = [1, 2, 4, 8, 16, 32]
N_DRAWS = 20                           # hospital draws per m
TARGETS = ["race_binary", "mortality", "gender_female", "age_ge65"]
ATOMS = ("red", "u_val", "u_str", "syn")

P12E = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
        "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}

cc = eicu.complete_case(labname_scope=P12E)
_, ecoh = eicu._pull()
hosp = ecoh.set_index("patientunitstayid")["hospitalid"].reindex(cc.groups).to_numpy()
demo, _ = eicu.demographic_targets()
vc = pd.Series(hosp).value_counts()

print(f"eICU matched-12 cohort n={len(cc.y):,} across {vc.size} hospitals")
print(f"largest hospital n={vc.max():,}\n")
print("eligible pool per m (hospitals holding >= 3000/m patients):")
for m in M_GRID:
    need = N_TOTAL // m
    print(f"  m={m:2d}  need n>={need:4d} per site  ->  {(vc >= need).sum():3d} eligible")

eICU matched-12 cohort n=120,266 across 198 hospitals
largest hospital n=3,803

eligible pool per m (hospitals holding >= 3000/m patients):
  m= 1  need n>=3000 per site  ->    3 eligible
  m= 2  need n>=1500 per site  ->   17 eligible
  m= 4  need n>= 750 per site  ->   54 eligible
  m= 8  need n>= 375 per site  ->  100 eligible
  m=16  need n>= 187 per site  ->  138 eligible
  m=32  need n>=  93 per site  ->  158 eligible


## 1. One codebook, fit on the pooled cohort, applied to every draw

Same discipline as `04_eicu_within_hospital`: the codebook is fit once per seed
on the full 120k cohort and every subset indexes into it. Only the composition
of the joint table varies across draws, never the channel definition.

In [2]:
def codebook(seed):
    v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(cc.Xv), K, seed=seed)
    s, _ = qz.kmeans_levels(qz.standardize_for_kmeans(cc.Xs, sentinel=eicu.TIME_SENTINEL),
                            K, seed=seed)
    return v, s

CODE = {sd: codebook(sd) for sd in SEEDS}
print(f"codebooks fit on n={len(cc.y):,} at k={K} -> {K*K*2} joint cells")


def target_mask_y(target):
    """Mortality uses the whole cohort; a demographic target drops rows where
    the contrast is undefined (the excluded ethnicity buckets, mainly)."""
    if target == "mortality":
        return np.ones(len(cc.y), bool), cc.y
    col = demo[target].reindex(pd.Index(cc.groups))
    m = col.notna().to_numpy()
    return m, col[m].to_numpy().astype(int)


def atoms(v, s, y):
    if len(np.unique(y)) < 2 or len(np.unique(v)) < 2 or len(np.unique(s)) < 2:
        return None
    d = pb.decompose(v, s, y, measure="broja")
    if not d["converged"]:
        return None
    a = d["atoms_pct_of_joint"]
    return {k: a[k] for k in ATOMS}

codebooks fit on n=120,266 at k=5 -> 50 joint cells


## 2. The dose-response

For each `m`: draw `m` distinct hospitals from those large enough, take
`3000/m` patients at random from each, and decompose under all three codebooks.
The RNG is seeded per (target, m, draw) so the notebook is deterministic.

In [3]:
perdraw = []
t0 = time.time()

for target in TARGETS:
    tmask, y_t = target_mask_y(target)
    h_t = hosp[tmask]
    idx_t = np.arange(tmask.sum())
    vc_t = pd.Series(h_t).value_counts()

    for m in M_GRID:
        per = N_TOTAL // m
        eligible = vc_t[vc_t >= per].index.to_numpy()
        if len(eligible) < m:
            print(f"  skip {target} m={m}: only {len(eligible)} eligible sites")
            continue

        for draw in range(N_DRAWS):
            # crc32, not hash(): Python randomises str hashing per process, so
            # hash() here would silently make the notebook non-reproducible.
            rng = np.random.default_rng(zlib.crc32(f"{target}|{m}|{draw}".encode()))
            sites = rng.choice(eligible, size=m, replace=False)
            take = []
            for h in sites:
                pool_h = idx_t[h_t == h]
                take.append(rng.choice(pool_h, size=per, replace=False))
            sel = np.concatenate(take)

            for sd in SEEDS:
                v_all, s_all = CODE[sd]
                a = atoms(v_all[tmask][sel], s_all[tmask][sel], y_t[sel])
                if a is None:
                    continue
                perdraw.append({"target": target, "m": m, "draw": draw, "seed": sd,
                                "n": int(len(sel)), "n_sites": int(m), **a})

    print(f"{target:15s} done  ({time.time()-t0:.0f}s cumulative)")

pdw = pd.DataFrame(perdraw)
print(f"\n{len(pdw)} draws total in {time.time()-t0:.0f}s")

race_binary     done  (10s cumulative)


mortality       done  (20s cumulative)


gender_female   done  (31s cumulative)


age_ge65        done  (40s cumulative)

1440 draws total in 40s


## 3. Trend per target

In [4]:
def trend(g):
    """OLS of u_str on log2(m): the slope is 'points of u_str lost per doubling
    of the number of contributing sites'. Spearman is reported alongside because
    the per-draw spread is large and the relationship need not be linear."""
    sl, ic, r, p, se = stats.linregress(np.log2(g.m), g.u_str)
    rho, prho = stats.spearmanr(g.m, g.u_str)
    mm = g.groupby("m").u_str.mean()
    first, last = mm.loc[mm.index.min()], mm.loc[mm.index.max()]
    return pd.Series({"slope_per_doubling": sl, "slope_p": p,
                      "spearman_rho": rho, "spearman_p": prho,
                      "m_first": first, "m_last": last,
                      "pct_removed": 100 * (first - last) / first if first else np.nan})

summary = pdw.groupby("target").apply(trend, include_groups=False)
means = pdw.pivot_table(index="target", columns="m", values="u_str", aggfunc="mean")
sds = pdw.pivot_table(index="target", columns="m", values="u_str", aggfunc="std")

print("mean u_str (% of joint information) by number of contributing hospitals\n")
print(means.to_string(float_format=lambda x: f"{x:.2f}"))
print("\nper-draw sd\n")
print(sds.to_string(float_format=lambda x: f"{x:.2f}"))
print("\ntrend\n")
print(summary.to_string(float_format=lambda x: f"{x:.3f}"))

mean u_str (% of joint information) by number of contributing hospitals

m                1     2     4     8    16    32
target                                          
age_ge65       0.05  0.12  0.52  0.05 0.09  0.04
gender_female -0.00 -0.00 -0.00 -0.00 0.01 -0.00
mortality      1.13  1.95  2.98  2.33 1.05  1.59
race_binary   12.47  5.86 11.91 11.87 4.55  3.48

per-draw sd

m                1    2     4     8    16   32
target                                        
age_ge65       0.16 0.35  2.61  0.28 0.41 0.17
gender_female  0.00 0.00  0.00  0.00 0.03 0.00
mortality      2.01 4.39  4.57  5.54 2.28 3.17
race_binary   11.54 6.53 13.83 13.43 8.12 7.67

trend

               slope_per_doubling  slope_p  spearman_rho  spearman_p  m_first  m_last  pct_removed
target                                                                                            
age_ge65                   -0.017    0.622        -0.041       0.441    0.046   0.037       19.883
gender_female               0.00

## 4. Is the m=1 anchor doing the work?

Only 3 hospitals are large enough to supply all 3,000 patients alone, so the
m=1 point is not a general statement about single sites. Refit the slope
dropping the low-m points, where the eligible pool is small.

This is the check that decides how much of §05 survives. A trend that holds on
m>=4 (54+ eligible sites) is a claim about pooling; one that only exists with
m=1 in the fit is a claim about three hospitals.

In [5]:
sens = []
for target, g in pdw.groupby("target"):
    for lo in (1, 2, 4, 8):
        gg = g[g.m >= lo]
        if gg.m.nunique() < 3:
            continue
        sl, ic, r, p, se = stats.linregress(np.log2(gg.m), gg.u_str)
        sens.append({"target": target, "m_min": lo, "n_m": gg.m.nunique(),
                     "slope_per_doubling": sl, "slope_p": p,
                     "sign_stable": bool(sl < 0)})
sens = pd.DataFrame(sens)
print("slope refit with low-m points dropped\n")
print(sens.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nverdict per target:")
for target, g in sens.groupby("target"):
    sig = (g.slope_p < 0.05)
    if sig.all():
        v = "ROBUST -- negative and significant at every truncation"
    elif not sig.any():
        v = "NO TREND at any truncation"
    else:
        v = ("UNSTABLE -- significance depends on which m are included "
             f"(p ranges {g.slope_p.min():.1e} to {g.slope_p.max():.2f})")
    print(f"  {target:15s} {v}")

slope refit with low-m points dropped

       target  m_min  n_m  slope_per_doubling  slope_p  sign_stable
     age_ge65      1    6              -0.017    0.622         True
     age_ge65      2    5              -0.059    0.234         True
     age_ge65      4    4              -0.141    0.068         True
     age_ge65      8    3              -0.006    0.836         True
gender_female      1    6               0.000    0.316        False
gender_female      2    5               0.001    0.420        False
gender_female      4    4               0.001    0.610        False
gender_female      8    3              -0.000    1.000         True
    mortality      1    6              -0.029    0.809         True
    mortality      2    5              -0.264    0.121         True
    mortality      4    4              -0.544    0.022         True
    mortality      8    3              -0.367    0.307         True
  race_binary      1    6              -1.398    0.000         True
  race_bi

## 5. Agreement with the surviving per-draw records

In [6]:
orig = pd.DataFrame(json.load(open("../out/_pooling_dose_perdraw.json")))
print("original run (code lost, draws survived) vs this reconstruction\n")
print(f"{'target':15s} {'m':>3s} {'orig':>7s} {'rebuilt':>8s} {'diff':>7s}")
for target in sorted(set(orig.target) & set(pdw.target)):
    for m in M_GRID:
        o = orig[(orig.target == target) & (orig.m == m)].u_str
        n = pdw[(pdw.target == target) & (pdw.m == m)].u_str
        if o.empty or n.empty:
            continue
        print(f"{target:15s} {m:3d} {o.mean():7.2f} {n.mean():8.2f} {n.mean()-o.mean():+7.2f}")

print("\nDifferences are expected: the original hospital draws and RNG seed were")
print("not recorded, so this resamples the same design rather than replaying it.")
print("What should agree is the DIRECTION and rough magnitude of the trend.")

original run (code lost, draws survived) vs this reconstruction

target            m    orig  rebuilt    diff
mortality         1    1.39     1.13   -0.26
mortality         2    1.09     1.95   +0.86
mortality         4    2.60     2.98   +0.38
mortality         8    2.02     2.33   +0.30
mortality        16    2.23     1.05   -1.18
mortality        32    0.77     1.59   +0.82
race_binary       1   11.54    12.47   +0.93
race_binary       2    7.45     5.86   -1.60
race_binary       4   10.20    11.91   +1.70
race_binary       8    6.63    11.87   +5.24
race_binary      16    3.63     4.55   +0.92
race_binary      32    2.66     3.48   +0.82

Differences are expected: the original hospital draws and RNG seed were
not recorded, so this resamples the same design rather than replaying it.
What should agree is the DIRECTION and rough magnitude of the trend.


## 6. Save

In [7]:
# The published file carries prose -- claim_tested, interpretation, caveats,
# framing_note -- that is the author's reasoning, not output. Carry it forward
# rather than overwriting it, and append what the rebuild established.
prev = json.load(open("../results/pooling_dose_response.json"))

payload = {
    "claim_tested": prev["claim_tested"],
    "design": (f"eICU-CRD, matched 12-analyte panel, k={K}, one shared codebook. TOTAL n "
               f"held fixed at {N_TOTAL:,} while the number of contributing hospitals m "
               f"varies over {M_GRID} (n/m drawn from each). {N_DRAWS} hospital draws x "
               f"{len(SEEDS)} codebook seeds per m."),
    "provenance": ("Rebuilt by notebooks/05_pooling_dose_response.ipynb. The original "
                   "generator was never saved; its per-draw records survive in "
                   "out/_pooling_dose_perdraw.json and every published statistic "
                   "recomputes from them exactly. Hospital ids and RNG seed were not "
                   "recorded, so this reproduces the design, not the draws."),
    # per target, not pooled: a demographic contrast drops rows, so its eligible
    # pool is strictly smaller than mortality's at the same m.
    "eligible_pool_per_m": {
        t: {f"m{m}": int((pd.Series(hosp[target_mask_y(t)[0]]).value_counts()
                          >= N_TOTAL // m).sum()) for m in M_GRID}
        for t in TARGETS},
    "dose_response_u_str": {
        t: {**{f"m{m}": float(means.loc[t, m]) for m in means.columns if not pd.isna(means.loc[t, m])},
            **{k: float(v) for k, v in summary.loc[t].items()}}
        for t in means.index},
    "truncation_sensitivity": sens.to_dict(orient="records"),
    # REWRITTEN 2026-08-29. The carried-forward text asserted that mortality's
    # atom is "NOT site-specific (p=0.94)" and that shrinkage under pooling is a
    # diagnostic separating local artefact from shared physiology. The
    # truncation sensitivity and the two-run sign disagreement below do not
    # support either, so both are withdrawn rather than reprinted.
    "verdict": ("SUPPORTED FOR RACE. NOT MEASURABLE FOR MORTALITY -- which is a "
                "statement about resolution, not about mortality being immune to pooling."),
    "interpretation": [
        "What this experiment establishes is that unique-to-structure differs "
        "enormously BY OUTCOME. At a single site race carries 10-12% of joint "
        "information in the structure channel, mortality 1-3%, and age and sex "
        "nothing that survives quantisation.",

        "For RACE the atom is site-specific and pooling removes it: 72-77% "
        "disappears going from one hospital to thirty-two at identical total n "
        "(slope -1.4 to -1.7 points per doubling, p<0.001), the trend stays "
        "negative and significant when the low-m points are dropped, and two "
        "independent runs of this design agree. That is consistent with race's "
        "structure signal being care-process and access -- routing, catchment, "
        "local ordering habit -- which differs between hospitals and averages "
        "out across them.",

        "For MORTALITY no pooling effect is resolvable, and this must NOT be read "
        "as the atom being universal or untouched. Mortality's u_str base is ~4x "
        "smaller than race's, per-draw sd is 2-5 points against a base near 1, the "
        "curve is non-monotone, and the two independent runs of this identical "
        "design disagree on the SIGN of the change (original: 45% removed; "
        "rebuild: 41% gained). Mortality carries too little non-physiological "
        "structure signal for a trend to be measured here. Less contamination, "
        "therefore no apparent pooling effect -- not pooling leaving it alone.",

        "WITHDRAWN: the earlier claim that 'whether an atom shrinks under pooling "
        "is a diagnostic' separating local practice from shared physiology. It "
        "required mortality's flatness to be a real null, and it is not. What "
        "survives is narrower and still worth having: a single-centre record can "
        "carry a large non-physiological component about a protected "
        "characteristic, and multi-site pooling demonstrably attenuates it.",
    ],
    "caveats": prev["caveats"] + [
        f"The eligible hospital pool changes with m and is very small at m=1: only "
        f"{int((vc >= N_TOTAL).sum())} hospitals hold >= {N_TOTAL:,} patients at all, and "
        "only 2 do for the race contrast. The m=1 anchor of the headline percentage is "
        "therefore a two-hospital average, one of which shows no u_str; m=32 draws from "
        "155. The site population varies along the x-axis, not only the site count.",
        "See truncation_sensitivity: the race trend holds when low-m points are dropped; "
        "the mortality 'flat' result does not, so the selectivity claim rests on a "
        "statistic whose sign depends on which m are included.",
    ],
    "framing_note": prev["framing_note"],
}
with open("../results/pooling_dose_response.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)
pdw.to_json("../out/_pooling_dose_perdraw_rebuilt.json", orient="records")
print("wrote results/pooling_dose_response.json")
print("wrote out/_pooling_dose_perdraw_rebuilt.json")

wrote results/pooling_dose_response.json
wrote out/_pooling_dose_perdraw_rebuilt.json
